# Step 2: VADER Sentiment Analysis
Analyzes review title + text sentiment using the offline VADER Sentiment Intensity Analyzer, validates alignment against star ratings, and generates diagnostics.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Hardcoded BASE_DIR
BASE_DIR = Path(r"C:\Users\Prashma\Desktop\Resume Projects\Data Analytics\NLP-powered customer review analytics system")
RAW_DATA_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DATA_DIR = BASE_DIR / "data" / "processed"

df = pd.read_csv(RAW_DATA_DIR / "amazon_reviews.csv")
print(f"Loaded {len(df):,} reviews.")


### 2.1 Calculate VADER Sentiment Scores

In [ ]:
analyzer = SentimentIntensityAnalyzer()
df["full_review"] = (df["review_title"].fillna("") + ". " + df["review_text"].fillna("")).str.strip()

compounds = []
labels = []
for text in df["full_review"]:
    vs = analyzer.polarity_scores(text)
    comp = vs["compound"]
    compounds.append(comp)
    if comp >= 0.05:
        labels.append("positive")
    elif comp <= -0.05:
        labels.append("negative")
    else:
        labels.append("neutral")

df["sentiment_compound"] = compounds
df["sentiment_label"] = labels
df.to_csv(PROCESSED_DATA_DIR / "reviews_with_sentiment.csv", index=False)
print("Saved reviews_with_sentiment.csv!")


### 2.2 Validation: Sentiment vs Star Rating Crosstab

In [ ]:
crosstab_pct = pd.crosstab(df["rating"], df["sentiment_label"], normalize="index") * 100
print(crosstab_pct.round(2).map(lambda x: f"{x:.2f}%"))

plt.figure(figsize=(8, 5))
sns.heatmap(crosstab_pct, annot=True, fmt=".1f", cmap="Blues", cbar=True)
plt.title("Sentiment Label vs Star Rating (Row %)", fontsize=13, fontweight="bold")
plt.xlabel("VADER Sentiment")
plt.ylabel("Star Rating")
plt.show()


### 2.3 Monthly Sentiment Trajectory

In [ ]:
df["month"] = pd.to_datetime(df["review_date"]).dt.to_period("M").dt.to_timestamp()
monthly_sentiment = df.groupby(["month", "sentiment_label"]).size().unstack(fill_value=0)
monthly_sentiment_pct = monthly_sentiment.div(monthly_sentiment.sum(axis=1), axis=0) * 100

plt.figure(figsize=(11, 5))
plt.plot(monthly_sentiment_pct.index, monthly_sentiment_pct["positive"], label="Positive %", color="#3ECF9E", marker="o", linewidth=2.5)
plt.plot(monthly_sentiment_pct.index, monthly_sentiment_pct["negative"], label="Negative %", color="#E5484D", marker="s", linewidth=2.5)
plt.plot(monthly_sentiment_pct.index, monthly_sentiment_pct["neutral"], label="Neutral %", color="#8B949E", marker="^", linewidth=2)
plt.title("Monthly Sentiment Trend (June 2024 - August 2025)", fontsize=13, fontweight="bold")
plt.ylabel("Percentage (%)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
